# Пример - Подпространства и преобразования графов

## Подготовка

Выполните все ячейки сверху вниз. Проверенная версия пакета: **0.9.0**.

In [1]:
from importlib.metadata import version, PackageNotFoundError
import subprocess
import sys

# Install only when the required package version is missing.
try:
    installed = version("polyform_algebra")
except PackageNotFoundError:
    installed = None
if installed != "0.9.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                          "https://github.com/magin67/polyform_algebra/archive/refs/tags/v0.9.0.zip"])


In [2]:
import polyform_algebra
assert polyform_algebra.get_version() == "0.9.0"
print("polyform_algebra", polyform_algebra.get_version())

polyform_algebra 0.9.0


На двух путях, звезде и двудольном графе проверим вычисления серии «Пространства». Начнём с метрики подпространства, координат и формы середины, затем сравним операции над графами.


Выполняйте фрагменты сверху вниз с версией 0.9.0. См. [Установка библиотеки и запуск примеров](https://magin67.github.io/pmg-notes/ru/pythonexamples/%D1%83%D1%81%D1%82%D0%B0%D0%BD%D0%BE%D0%B2%D0%BA%D0%B0-%D0%B1%D0%B8%D0%B1%D0%BB%D0%B8%D0%BE%D1%82%D0%B5%D0%BA%D0%B8-%D0%B8-%D0%B7%D0%B0%D0%BF%D1%83%D1%81%D0%BA-%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80%D0%BE%D0%B2).


## Метрика выбранного подпространства

Возьмём путь $a-b-c$ с единичными проводимостями и векторы $u=b-a$, $v=c-b$. Пара вершин $a,c$ задаёт направление $c-a=u+v$. Нормы этого вектора и границы $(abc)$ вычислим через нормированные потенциалы метрической полиформы и проверим через матрицу Грина.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0/%D0%B1%D0%B0%D0%B7%D0%B8%D1%81#%D0%BF%D1%80%D1%8F%D0%BC%D0%BE%D0%B9-%D1%80%D0%B0%D1%81%D1%87%D1%91%D1%82-%D0%BD%D0%B0-%D0%BF%D1%83%D1%82%D0%B8).

In [3]:
from fractions import Fraction as Q
from sympy import Matrix
from polyform_algebra.core import (
    INFINITY, Atom, Boundary, Context, boundary, contract, coordinates,
    dirichlet, equivalent, evaluate_form, form_exp, gram, induced_subgraph,
    kron_reduce, laplacian, laplacian_form, network, polar_form, potential,
    resistance, spanning_coefficient, spanning_tree_weight, square, wedge,
)

def difference(net, start, end):
    return coordinates(net.context, {
        label: int(label == end) - int(label == start)
        for label in net.context.labels
    })

unit_path = network(("a", "b", "c"), (("a", "b", 1), ("b", "c", 1)))
u = difference(unit_path, "a", "b")
v = difference(unit_path, "b", "c")
assert gram(unit_path, (u, v)).value.values == ((1, 0), (0, 1))

metric = form_exp(laplacian_form(unit_path).value).value
a_index, b_index, c_index = (unit_path.context.labels.index(label) for label in ("a", "b", "c"))
edge_form = square(Boundary(unit_path.context, (a_index, c_index))).value
triangle_form = square(Boundary(unit_path.context, (a_index, b_index, c_index))).value
tau = spanning_coefficient(metric).value
line_norm = potential(metric, edge_form).value / tau
boundary_norm = potential(metric, triangle_form).value / tau
assert tau == 1
assert line_norm == evaluate_form(unit_path, edge_form).value == 2
assert boundary_norm == evaluate_form(unit_path, triangle_form).value == 1
print("R_ac =", line_norm, "| (abc)^2 =", boundary_norm)

R_ac = 2 | (abc)^2 = 1


## Би-координаты и ди-координаты

На том же пути $u,v$ ортогональны и имеют единичную норму. Возьмём базис $h_1=u$, $h_2=u+v$ и объект $v$. Би-координаты $\mathbf b$ задают его разложение, ди-координаты $\mathbf d$ равны скалярным произведениям с базисными векторами. Пакет вычисляет грамиан и ди-координаты, а SymPy решает систему $\mathbf G\mathbf b=\mathbf d$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0/%D0%B1%D0%B8-%D0%BA%D0%BE%D0%BE%D1%80%D0%B4%D0%B8%D0%BD%D0%B0%D1%82%D1%8B-%D0%B8-%D0%B4%D0%B8-%D0%BA%D0%BE%D0%BE%D1%80%D0%B4%D0%B8%D0%BD%D0%B0%D1%82%D1%8B#%D0%BF%D1%80%D0%B8%D0%BC%D0%B5%D1%80-%D0%BD%D0%B5%D0%BE%D1%80%D1%82%D0%BE%D0%B3%D0%BE%D0%BD%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D0%B3%D0%BE-%D0%B1%D0%B0%D0%B7%D0%B8%D1%81%D0%B0).

In [4]:
basis = (u, u + v)
G = Matrix(gram(unit_path, basis).value.values)
di = Matrix(gram(unit_path, basis, (v,)).value.values)
bi = G.LUsolve(di)
reciprocal = (u - v, v)

assert tuple(bi) == (-1, 1) and tuple(di) == (0, 1)
assert bi[0] * basis[0] + bi[1] * basis[1] == v
assert gram(unit_path, basis, reciprocal).value.values == ((1, 0), (0, 1))
print("G =", G.tolist(), "| b =", tuple(bi), "| d =", tuple(di))

G = [[1, 1], [1, 2]] | b = (-1, 1) | d = (0, 1)


## Форма середины

Пусть $q=(a+c)/2$. Пакет вычисляет полярную форму $\{q,q\}=2[q]^2$. Сравним её половину с разностью атомной и разностной частей. Проверка сравнивает формальные коэффициенты до выбора метрики.

$$
[q]^2=\frac12[a]^2+\frac12[c]^2-\frac14[(ac)]^2
$$

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0/%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5-%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BD%D0%B0-%D0%B0%D1%82%D0%BE%D0%BC%D0%BD%D1%83%D1%8E-%D0%B8-%D1%80%D0%B0%D0%B7%D0%BD%D0%BE%D1%81%D1%82%D0%BD%D1%83%D1%8E-%D1%87%D0%B0%D1%81%D1%82%D0%B8#%D1%80%D0%B0%D0%B7%D0%BB%D0%BE%D0%B6%D0%B5%D0%BD%D0%B8%D0%B5-%D1%84%D0%BE%D1%80%D0%BC%D1%8B-%D0%BF%D0%BE-%D1%8D%D0%BB%D0%B5%D0%BC%D0%B5%D0%BD%D1%82%D0%B0%D0%BC-%D1%84%D1%80%D0%B5%D0%B9%D0%BC%D0%B0-%D0%B8-%D0%B8%D1%85-%D1%80%D0%B0%D0%B7%D0%BD%D0%BE%D1%81%D1%82%D1%8F%D0%BC).

In [5]:
context = unit_path.context
a = Atom(context, a_index)
c = Atom(context, c_index)
q = (a.as_exterior() + c.as_exterior()) * Q(1, 2)
midpoint_square = polar_form(q, q).value * Q(1, 2)
atomic = (square(a).value + square(c).value) * Q(1, 2)
difference_part = edge_form * Q(1, 4)
assert equivalent(midpoint_square, atomic - difference_part).value
print("[q]^2 = ([a]^2 + [c]^2)/2 - [(ac)]^2/4")

[q]^2 = ([a]^2 + [c]^2)/2 - [(ac)]^2/4


## Граница и аффинный вес

Зададим аффинные веса $\epsilon(a)=\epsilon(b)=1$, $\epsilon(z)=0$. Из правила Лейбница получаем $\partial z=0$ и $\partial(az)=z$. В этом фрагменте контекст задаёт аффинный вес. Свойство $z^2=0$ в изотропной метрике требует отдельно заданного скалярного произведения.

[Заметка о нуль-векторе](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0/%D0%BD%D1%83%D0%BB%D1%8C-%D0%B2%D0%B5%D0%BA%D1%82%D0%BE%D1%80,-%D0%B8%D0%B7%D0%BE%D1%82%D1%80%D0%BE%D0%BF%D0%BD%D0%BE%D0%B5-%D0%BF%D1%80%D0%B5%D0%B4%D1%81%D1%82%D0%B0%D0%B2%D0%BB%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B8-%D0%B3%D1%80%D0%B0%D0%BD%D0%B8%D1%86%D0%B0).

In [6]:
affine_context = Context(("a", "b", "z"), epsilon=(1, 1, 0))
a = Atom(affine_context, affine_context.labels.index("a"))
z = Atom(affine_context, affine_context.labels.index("z"))
az = wedge(a, z).value

assert not boundary(z).value.coefficients
assert boundary(az).value == z.as_exterior()
assert not boundary(boundary(az).value).value.coefficients
print("∂z = 0; ∂(az) = z")

∂z = 0; ∂(az) = z


## Задача Дирихле на взвешенном пути

На пути $a-x-y-b$ с проводимостями $2,1,3$ закрепим значения $f_a=0$, $f_b=11$. Матрица $\mathbf B$ показывает, как значения в $a,b$ влияют на $f_x,f_y$. Проверим решение внутренними уравнениями токов и посчитаем энергию как сумму по рёбрам.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0/%D0%B2%D1%8B%D0%B4%D0%B5%D0%BB%D0%B5%D0%BD%D0%BD%D1%8B%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D1%8B-%D0%B8-%D0%B7%D0%B0%D0%B4%D0%B0%D1%87%D0%B0-%D0%B4%D0%B8%D1%80%D0%B8%D1%85%D0%BB%D0%B5#%D0%BC%D0%B0%D1%82%D1%80%D0%B8%D1%87%D0%BD%D0%B0%D1%8F-%D0%B7%D0%B0%D0%BF%D0%B8%D1%81%D1%8C-%D0%B4%D0%BB%D1%8F-%D0%B2%D1%8B%D1%87%D0%B8%D1%81%D0%BB%D0%B5%D0%BD%D0%B8%D0%B9).

In [7]:
path = network(("a", "x", "y", "b"), (("a", "x", 2), ("x", "y", 1), ("y", "b", 3)))
selected = ("a", "b")
solution = dirichlet(path, selected).value
values = solution.extend({"a": 0, "b": 11}).value
f = dict(zip(path.context.labels, values.values))
energy = sum(c * (f[u] - f[v])**2 for u, v, c in path.edges)
net_currents = Matrix(laplacian(path).value.values) * Matrix(values.values)

assert solution.I == ("x", "y")
assert solution.B.values == ((Q(8, 11), Q(3, 11)), (Q(2, 11), Q(9, 11)))
assert f == {"a": 0, "x": 3, "y": 9, "b": 11}
assert tuple(net_currents) == (-6, 0, 0, 6)
assert energy == 66
print("B =", [[str(value) for value in row] for row in solution.B.values])
print("f:", ", ".join(f"{label}={value}" for label, value in f.items()))
print("Energy =", energy)

B = [['8/11', '3/11'], ['2/11', '9/11']]
f: a=0, x=3, y=9, b=11
Energy = 66


## Удаление, стягивание и исключение

На том же пути оставим вершины $a,b$ и сравним три операции. Удаление $x,y$ разрывает путь. Стягивание отождествляет $x$ с $a$, а $y$ с $b$. Исключение по Крону сохраняет резистенс между $a,b$.

| Операция | Проводимость $ab$ | Резистенс $R_{ab}$ |
| --- | ---: | ---: |
| Удаление | Связь отсутствует | $\infty$ |
| Стягивание | $1$ | $1$ |
| Исключение | $6/11$ | $11/6$ |

Стягивание также позволяет сочленить независимые рёбра $ab$ и $cd$ в общей вершине $s$. Для проводимостей $2$ и $3$ остовной коэффициент нового пути равен $6$, а резистенс между концами равен $1/2+1/3=5/6$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0/%D1%81%D1%82%D1%8F%D0%B3%D0%B8%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD-%D0%B8-%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B3%D1%80%D0%B0%D1%84%D0%BE%D0%B2#%D1%81%D0%BE%D1%87%D0%BB%D0%B5%D0%BD%D0%B5%D0%BD%D0%B8%D0%B5-%D0%BF%D0%BE-%D0%BE%D0%B4%D0%BD%D0%BE%D0%B9-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD%D0%B5).

In [8]:
deleted = induced_subgraph(path, selected).value
contracted = contract(path, {"a": ("a", "x"), "b": ("y", "b")}).value.network
reduced = kron_reduce(path, selected).value.network

def pair_resistance(net, start, end):
    return resistance(net, pairs=((start, end),)).value.values[0]

assert deleted.edges == () and pair_resistance(deleted, "a", "b") == INFINITY
assert contracted.edges == (("a", "b", 1),)
assert reduced.edges == (("a", "b", Q(6, 11)),)
assert pair_resistance(contracted, "a", "b") == 1
assert pair_resistance(reduced, "a", "b") == pair_resistance(path, "a", "b") == Q(11, 6)
print("R_ab:", {
    "deletion": "∞",
    "contraction": str(pair_resistance(contracted, "a", "b")),
    "elimination": str(pair_resistance(reduced, "a", "b")),
})

separate = network(("a", "b", "c", "d"), (("a", "b", 2), ("c", "d", 3)))
glued = contract(separate, {"a": ("a",), "s": ("b", "c"), "d": ("d",)}).value.network
assert spanning_tree_weight(glued).value == 6
assert pair_resistance(glued, "a", "d") == Q(5, 6)
print("a–s–d: tau =", spanning_tree_weight(glued).value, "| R_ad =", pair_resistance(glued, "a", "d"))

R_ab: {'deletion': '∞', 'contraction': '1', 'elimination': '11/6'}
a–s–d: tau = 6 | R_ad = 5/6


## Исключение центра звезды

Исключим центр $x$ звезды с листьями $a,b,c$ и единичными проводимостями. Получим треугольник, где каждая наведённая связь имеет проводимость $1/3$. Проверим сохранение резистенса $2$ для каждой пары листьев и остовной коэффициент треугольника $1/3$.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0/%D0%B8%D1%81%D0%BA%D0%BB%D1%8E%D1%87%D0%B5%D0%BD%D0%B8%D0%B5-%D0%B2%D0%B5%D1%80%D1%88%D0%B8%D0%BD-%D0%B8-%D0%BD%D0%B0%D0%B2%D0%B5%D0%B4%D1%91%D0%BD%D0%BD%D1%8B%D0%B5-%D1%81%D0%B2%D1%8F%D0%B7%D0%B8#%D0%B7%D0%B2%D0%B5%D0%B7%D0%B4%D0%B0-%D0%BF%D1%80%D0%B5%D0%B2%D1%80%D0%B0%D1%89%D0%B0%D0%B5%D1%82%D1%81%D1%8F-%D0%B2-%D1%82%D1%80%D0%B5%D1%83%D0%B3%D0%BE%D0%BB%D1%8C%D0%BD%D0%B8%D0%BA).

In [9]:
leaves = ("a", "b", "c")
star = network(leaves + ("x",), tuple((leaf, "x", 1) for leaf in leaves))
triangle = kron_reduce(star, leaves).value.network

assert triangle.edges == (("a", "b", Q(1, 3)), ("a", "c", Q(1, 3)), ("b", "c", Q(1, 3)))
for start, end, _ in triangle.edges:
    assert pair_resistance(star, start, end) == pair_resistance(triangle, start, end) == 2
assert spanning_tree_weight(triangle).value == Q(1, 3)
print(", ".join(f"c_{u}{v}={weight}" for u, v, weight in triangle.edges), "| R_ab = 2")

c_ab=1/3, c_ac=1/3, c_bc=1/3 | R_ab = 2


## Две доли двудольного графа

Возьмём таблицу участий сотрудников $a,b$ в документах $x,y$.

| Сотрудник | $x$ | $y$ |
| --- | ---: | ---: |
| $a$ | $1$ | $1$ |
| $b$ | $2$ | $0$ |

Нулю соответствует отсутствие связи. Если сохранить сотрудников, получим проводимость $c_{ab}=2/3$. Если сохранить документы, получим $c_{xy}=1/2$. Каждое исключение сохраняет резистенсы выбранной доли.

[Соответствующий раздел заметки](https://magin67.github.io/pmg-notes/ru/%D1%80%D0%B5%D0%B7%D0%B8%D1%81%D1%82%D0%B8%D0%B2%D0%BD%D0%B0%D1%8F-%D0%B3%D0%B5%D0%BE%D0%BC%D0%B5%D1%82%D1%80%D0%B8%D1%8F-%D0%B8-%D0%B4%D1%83%D0%B0%D0%BB%D1%8C%D0%BD%D0%BE%D1%81%D1%82%D1%8C/%D0%BF%D1%80%D0%BE%D1%81%D1%82%D1%80%D0%B0%D0%BD%D1%81%D1%82%D0%B2%D0%B0/%D1%81%D0%B2%D1%8F%D0%B7%D0%B8-%D0%B2%D0%BD%D1%83%D1%82%D1%80%D0%B8-%D0%BE%D0%B4%D0%BD%D0%BE%D0%B9-%D0%B4%D0%BE%D0%BB%D0%B8-%D0%B4%D0%B2%D1%83%D0%B4%D0%BE%D0%BB%D1%8C%D0%BD%D0%BE%D0%B3%D0%BE-%D0%B3%D1%80%D0%B0%D1%84%D0%B0#%D0%B4%D0%B2%D0%B0-%D0%BD%D0%B0%D0%B2%D0%B5%D0%B4%D1%91%D0%BD%D0%BD%D1%8B%D1%85-%D0%B3%D1%80%D0%B0%D1%84%D0%B0).

In [10]:
people = ("a", "b")
documents = ("x", "y")
participation = network(people + documents, (("a", "x", 1), ("a", "y", 1), ("b", "x", 2)))
people_graph = kron_reduce(participation, people).value.network
document_graph = kron_reduce(participation, documents).value.network

assert people_graph.edges == (("a", "b", Q(2, 3)),)
assert document_graph.edges == (("x", "y", Q(1, 2)),)
assert pair_resistance(people_graph, "a", "b") == pair_resistance(participation, "a", "b") == Q(3, 2)
assert pair_resistance(document_graph, "x", "y") == pair_resistance(participation, "x", "y") == 2
print("c_ab =", people_graph.edges[0][2], "| c_xy =", document_graph.edges[0][2])

c_ab = 2/3 | c_xy = 1/2
